# L2 — Number partitioning: squaring a difference

**2026 Quantum Accelerator · Day 1 · QUBO ladder · step 2 of 5 · ~15 minutes**

**New on this step:** the objective is the *square of a linear expression*. Squaring creates
every pairwise product at once (so Q is dense), and `x_i² = x_i` turns the diagonal of the
square into linear terms. The same mechanism later turns equality constraints into penalties;
here the "constraint" *is* the objective.

**The problem.** Split a set of numbers into two groups with equal sums. It is "the easiest
hard problem": NP-hard, yet dynamic programming solves it when the numbers are small. **Plan.**
The slide instance by hand, then 40 numbers at three levels of difficulty.

---
## 1. The slide instance

In [1]:
import numpy as np
import helpers_ladder as h

a = np.array([3, 5, 8])                  # <-- CHANGE ME: try [3, 5, 9] (odd total: best difference 1)
labels = ["x1", "x2", "x3"]              # x_i = 1 means "number i goes in set A"
T = a.sum()
print(f"S = {a.tolist()}, total T = {T}, a perfect split has each side equal to {T/2:g}")

S = [3, 5, 8], total T = 16, a perfect split has each side equal to 8


---
## 2. Q by hand

In [2]:
Q = np.outer(a, a) - np.diag(a * T)      # Q_ii = a_i (a_i - T),  Q_ij = a_i a_j
h.print_matrix(Q, labels, "Q (minimise x^T Q x)")

Q (minimise x^T Q x)
        x1   x2   x3
   x1  -39   15   24
   x2   15  -55   40
   x3   24   40  -64


**Where it comes from.** Write the difference as `Σ a_i s_i` with `s_i = 2x_i − 1`; square it,
divide by 4, and use `x_i² = x_i` to fold the squares onto the diagonal. What is left is
`(Σ a_i x_i)² − T·Σ a_i x_i`. Its minimum is `−T²/4` (here −64), reached exactly when the split
is perfect.

---
## 3. Every answer

In [3]:
table = h.exact_table(Q, labels)
table["difference"] = [abs(2 * int(a @ row.values[:3]) - T) for _, row in table.iterrows()]
table

,x1,x2,x3,energy,objective,difference
0,0,0,1,-64,-64,0
1,1,1,0,-64,-64,0
2,0,1,0,-55,-55,6
3,1,0,1,-55,-55,6
4,0,1,1,-39,-39,10
5,1,0,0,-39,-39,10
6,0,0,0,0,0,16
7,1,1,1,0,0,16


**What you see.** The two rows at −64 are the same split with the groups swapped (the same
flip symmetry as in L1), and both have difference 0. Worse splits sit higher in energy.

---
## 4. The same Q from PyQUBO

In [4]:
from pyqubo import Binary

x = [Binary(f"x{i+1}") for i in range(3)]
diff = sum(int(ai) * (2 * xi - 1) for ai, xi in zip(a, x))
model = (diff ** 2).compile()
qubo, offset = model.to_qubo()
Q_lib = h.symmetric_from_qubo_dict(qubo, labels)
print(np.allclose(4 * Q, Q_lib), offset)     # True 256.0

True 256.0


**What you see.** Two differences from our matrix, and both are legitimate. PyQUBO did not
divide by 4, so its matrix is `4·Q`. And it kept the constant `T² = 256`, so its energies are
`4·(ours) + 256`: its minimum 0 corresponds to our −64. This is the first time you see that two
correct QUBOs of one problem can differ by a scale and a constant.

---
## 5. Now too big for a slide: forty numbers, three difficulty levels

In [5]:
rows = h.partition_sweep(n=40, max_values=(100, 10_000, 1_000_000), seed=1)   # <-- CHANGE ME
print(f"{'numbers up to':>14}{'T':>12}{'DP best':>9}{'SA best':>9}{'reads at DP value':>19}{'SA time':>10}{'DP time':>10}")
for r in rows:
    print(f"{r['max_value']:>14,}{r['T']:>12,}{r['dp_diff']:>9}{r['sa_best_diff']:>9}"
          f"{r['hit_pct']:>18.0f}%{r['seconds']:>8.2f} s{r['dp_seconds']:>8.2f} s")
h.plot_partition_sweep(rows)

 numbers up to           T  DP best  SA best  reads at DP value   SA time   DP time
           100       1,992        0        0                30%    0.14 s    0.00 s
        10,000     197,283        1        1                 2%    0.16 s    0.00 s
     1,000,000  19,726,278        0       74                 0%    0.15 s    0.08 s


**What you see.** Same n, same solver, same budget — only the number of bits per number
changed. With small numbers there are astronomically many perfect splits and the annealer
trips over one. With 20-bit numbers a perfect split is a needle in a haystack (Mertens'
phase transition sits at about one bit per number) and the annealer stalls at 74 while the
ten-line DP still finds 0.

Two lessons. (1) "Unconstrained" does not mean "easy for annealing". (2) The Q entries are now
`a_i·a_j ≈ 10¹²`, a dynamic range no analog annealer can represent: the precision cost of
squaring. DP wins here because `T` is small *relative to 2ⁿ*, not because the problem is easy in general.

---
## 6. What changed on this step

- The objective is a square: every pair interacts, so Q is **dense** (L1's Q was as sparse as the graph).
- The diagonal comes from `x² = x`: linear terms appear from squaring.
- Two correct QUBOs can differ by a scale (÷4) and a constant (T²).
- Same n, harder numbers: success fell from 30 % to 0 %; large coefficients are a cost of their own.
- Next, **L3**: coefficients come from logic, signs are mixed, and the constant means something.

---
## Stretch

**A.** Use `num_sweeps=200_000` on the 10⁶ level. How far does SA get, and at what cost in time?

**B.** n = 40 with numbers up to 10⁶ has 2⁴⁰ splits. Why can DP still finish?

**C.** Multiway partitioning into 3 groups: how many variables would a one-hot encoding need,
and which new constraint appears? (Bridge to L5 and to notebook 04.)